In [21]:
## Run one time
import os
from PIL import Image

src_root = 'dataset'  # This folder contains Open_eyes_subset and Blinks_subset
dst_root = 'resized_dataset'

# Create destination root folder
os.makedirs(dst_root, exist_ok=True)

# Walk through source folder
for subdir, _, files in os.walk(src_root):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            src_path = os.path.join(subdir, file)

            # Get relative path from source root and construct destination path
            rel_path = os.path.relpath(src_path, src_root)
            dst_path = os.path.join(dst_root, rel_path)

            # Create destination subdirectory if it doesn't exist
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)

            # Resize and save
            try:
                with Image.open(src_path) as img:
                    img = img.resize((224, 224))
                    img.save(dst_path)
            except Exception as e:
                print(f"Error processing {src_path}: {e}")


In [23]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Load the MobileNetV2 base model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))



In [25]:
# Freeze all layers in the base model
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)  # Binary classification

model = Model(inputs=base_model.input, outputs=predictions)



In [27]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

### Done till this part

In [9]:
# import os
# import shutil
# from pathlib import Path

# # Source folders
# open_src = 'Open_eyes_subset'
# blink_src = 'Blinks_subset'

# # Target folders
# target_base = 'dataset'
# open_target = os.path.join(target_base, 'open')
# blink_target = os.path.join(target_base, 'blink')

# # Create target directories if they don't exist
# os.makedirs(open_target, exist_ok=True)
# os.makedirs(blink_target, exist_ok=True)

# def flatten_images(src_folder, target_folder, label):
#     count = 0
#     for root, _, files in os.walk(src_folder):
#         for file in files:
#             if file.lower().endswith(('.jpg', '.jpeg', '.png')):
#                 src_path = os.path.join(root, file)
#                 new_filename = f"{label}_{count}.jpg"
#                 dst_path = os.path.join(target_folder, new_filename)
#                 shutil.copy2(src_path, dst_path)
#                 count += 1
#     print(f"Moved {count} images to {target_folder}")

# flatten_images(open_src, open_target, label='open')
# flatten_images(blink_src, blink_target, label='blink')


Moved 553 images to dataset/open
Moved 995 images to dataset/blink


In [29]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)


In [31]:
train_generator = train_datagen.flow_from_directory(
    'resized_dataset',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    'resized_dataset',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)


Found 1239 images belonging to 2 classes.
Found 309 images belonging to 2 classes.


In [33]:
# Train the model (only the last layer)
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10
)

#EarlyStopping
#ModelCheckpoint


Epoch 1/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 20s 464ms/step - accuracy: 0.7597 - loss: 0.3953 - val_accuracy: 0.9806 - val_loss: 0.1067
Epoch 2/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 18s 460ms/step - accuracy: 0.9927 - loss: 0.0425 - val_accuracy: 0.9806 - val_loss: 0.0812
Epoch 3/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 18s 452ms/step - accuracy: 0.9979 - loss: 0.0203 - val_accuracy: 0.9806 - val_loss: 0.0952
Epoch 4/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 17s 450ms/step - accuracy: 0.9975 - loss: 0.0154 - val_accuracy: 0.9806 - val_loss: 0.0715
Epoch 5/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 18s 452ms/step - accuracy: 0.9988 - loss: 0.0077 - val_accuracy: 0.9871 - val_loss: 0.0306
Epoch 6/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 18s 456ms/step - accuracy: 0.9991 - loss: 0.0072 - val_accuracy: 0.9871 - val_loss: 0.0278
Epoch 7/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 18s 453ms/step - accuracy: 0.9994 - loss: 0.0046 - val_accuracy: 0.9871 - val_loss: 0.0433
Epoch 8/10
39/39 ━━━━━━━━━━━━━━━━━━━━ 18s 456ms/step - accuracy: 0.9987 - loss: 0.0037 - val_accu